# 03 subtype grid summary

이 노트북은 sweep 결과를 비교하고 좋은 run 후보를 고르는 용도다.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
if (PROJECT_ROOT / "src").exists():
    sys.path.append(str(PROJECT_ROOT / "src"))
else:
    PROJECT_ROOT = Path.cwd().parent
    sys.path.append(str(PROJECT_ROOT / "src"))

from subtype.io_utils import load_run_outputs


## 1. sweep run root 설정

In [ ]:
GRID_ROOT = PROJECT_ROOT / "data" / "20260309_pilot" / "results" / "subtype_discovery" / "grid_runs_small"
SUMMARY_CSV = GRID_ROOT / "grid_summary.csv"

summary = pd.read_csv(SUMMARY_CSV)
summary.head()


## 2. best silhouette 기준 정렬

In [ ]:
summary_sorted = summary.sort_values(["best_silhouette", "n_features"], ascending=[False, True])
display(summary_sorted.head(20))


## 3. silhouette / feature 수 quick plot

In [ ]:
plt.figure(figsize=(6,4))
plt.scatter(summary["n_features"], summary["best_silhouette"], s=40)
plt.xlabel("n_features")
plt.ylabel("best_silhouette")
plt.title("Grid runs: silhouette vs n_features")
plt.show()


## 4. 파라미터별 분포 보기

In [ ]:
for col in summary.columns:
    if col in {"run_name", "returncode", "selected_feature_names"}:
        continue
    if summary[col].nunique() <= 8:
        print(f"\n=== {col} ===")
        display(summary.groupby(col)["best_silhouette"].agg(["count", "mean", "max"]).sort_values("mean", ascending=False))


## 5. 상위 run 하나 골라서 상세 보기

In [ ]:
top_run_name = summary_sorted.iloc[0]["run_name"]
top_run_dir = GRID_ROOT / top_run_name
top_run_name, top_run_dir


In [ ]:
top_run = load_run_outputs(top_run_dir)

display(pd.DataFrame([top_run["metrics"]]) if top_run["metrics"] is not None else pd.DataFrame())
display(pd.DataFrame([top_run["cohort_info"]]) if top_run["cohort_info"] is not None else pd.DataFrame())
display(pd.DataFrame({"selected_feature": top_run["selected_features"]["selected_features"]})
        if top_run["selected_features"] is not None else pd.DataFrame())


In [ ]:
if top_run["patient_pca"] is not None and top_run["patient_features_batch_adjusted"] is not None:
    plot_df = top_run["patient_pca"].merge(
        top_run["patient_features_batch_adjusted"][["analysis_id", "source"]],
        on="analysis_id",
        how="left"
    )
    plt.figure(figsize=(5,4))
    for s in sorted(plot_df["source"].dropna().unique()):
        sub = plot_df[plot_df["source"] == s]
        plt.scatter(sub["PC1"], sub["PC2"], s=35, label=s)
    plt.xlabel("PC1"); plt.ylabel("PC2"); plt.title(f"{top_run_name}: PCA by source")
    plt.legend()
    plt.show()


In [ ]:
if top_run["patient_umap"] is not None and top_run["patient_clusters"] is not None:
    plot_df = top_run["patient_umap"].merge(top_run["patient_clusters"], on="analysis_id", how="left")
    plt.figure(figsize=(5,4))
    for s in sorted(plot_df["patient_cluster"].dropna().unique()):
        sub = plot_df[plot_df["patient_cluster"] == s]
        plt.scatter(sub["UMAP1"], sub["UMAP2"], s=35, label=f"C{s}")
    plt.xlabel("UMAP1"); plt.ylabel("UMAP2"); plt.title(f"{top_run_name}: UMAP by cluster")
    plt.legend()
    plt.show()


## 6. 여러 상위 run 비교

In [ ]:
top_runs = summary_sorted.head(5)["run_name"].tolist()
compare_rows = []

for run_name in top_runs:
    run_dir = GRID_ROOT / run_name
    run = load_run_outputs(run_dir)
    metrics = run["metrics"] or {}
    row = {"run_name": run_name, **metrics}
    if run["selected_features"] is not None:
        row["n_selected_features_json"] = len(run["selected_features"].get("selected_features", []))
    compare_rows.append(row)

compare_df = pd.DataFrame(compare_rows)
display(compare_df)


## 7. 추천 선별 기준

- `best_silhouette`가 너무 낮지 않을 것
- `n_features`가 지나치게 많지 않을 것
- healthy만 분리하는 run이 아니라 disease 내부 구조도 남을 것
- selected feature가 biology적으로 해석 가능할 것